## Week 2: Vector Search — Part 3: Using `sqlitesearch` as persistent knowledge base (instead of `minsearch`)

* Ingest data and persistent it in a sqlite knowledge base
* Replaces nearest neighbour (NN) search with Approximate nearest neighbour (ANN) search to reduce search latency 

```text
NN (exact):    compare query against ALL documents -> top 5
ANN (approx):  narrow down to a region -> compare within region -> top 5
```

### Setup: load model and data

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

documents = load_faq_data()
index = build_index(documents)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
from tqdm.auto import tqdm
import numpy as np

texts = [doc["question"] + " " + doc["answer"] for doc in documents]

vectors = []
batch_size = 50

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i : i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)
print(f"Embeddings matrix shape: {X.shape}")

  0%|          | 0/28 [00:00<?, ?it/s]

Embeddings matrix shape: (1368, 384)


### Build the sqlitesearch index

sqlitesearch supports three ANN modes:

- `lsh` (default): up to 100K vectors, random hyperplane projections
- `ivf`: 10K-500K vectors, K-means clustering (approx Nearest neighbours)
- `hnsw`: 10K-1M+ vectors, proximity graph (highest recall)

For our small dataset, `lsh` is fine. All modes use two-phase search: approximate candidate retrieval, then exact cosine similarity
reranking.

In [4]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"], mode="ivf", db_path="faq_vectors2.db"
)

vs_index.fit(vectors, documents)

In [ ]:
query = "when did course start"
query_vector = model.encode(query)

results = vs_index.search(
    query_vector, num_results=5, filter_dict={"course": "llm-zoomcamp"}
)

In [10]:
results

[{'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': 'aa310de435',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Can I run the course locally instead of Codespaces?',
  'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\n\nIf you run locally, make sure you document your setup and keep your environment reproducible.'},
 {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '85384a18e5',
  

In [11]:
vs_index.close()  # close the connection to this sqlite db